In [2]:
import requests, json
import pandas as pd
from tabulate import tabulate

ModuleNotFoundError: No module named 'tabulate'

In [ ]:
def RREO(an_exercicio, nr_periodo, co_tipo_demonstrativo, no_anexo, id_ente):
    url = f'https://apidatalake.tesouro.gov.br/ords/siconfi/tt//rreo?an_exercicio={an_exercicio}&nr_periodo={nr_periodo}&co_tipo_demonstrativo={co_tipo_demonstrativo}&no_anexo={no_anexo}&co_esfera=&id_ente={id_ente}'
    response = requests.get(url)
    if(response.status_code != 200): 
            return({result:'Falhou'})
    dados = response.json()
    return dados

In [ ]:
#RREO Anexo 01

no_anexo = 'RREO-Anexo+01'

dados = RREO(an_exercicio, nr_periodo, co_tipo_demonstrativo, no_anexo, id_ente)

rreo_df = dados['items']
rreo_df = pd.DataFrame(rreo_df)

rreo_df = rreo_df.pivot(index=['conta', 'cod_conta'], columns='coluna', values='valor').reset_index()

# Reordenando as colunas conforme a ordem especificada
colunas_ordenadas = [
    'conta', 'cod_conta', 'PREVISÃO INICIAL', 'PREVISÃO ATUALIZADA (a)', 'No Bimestre (b)',
    '% (b/a)', 'Até o Bimestre (c)', '% (c/a)', 'SALDO (a-c)', 'DOTAÇÃO INICIAL (d)',
    'DOTAÇÃO ATUALIZADA (e)', 'DESPESAS EMPENHADAS NO BIMESTRE', 'DESPESAS EMPENHADAS ATÉ O BIMESTRE (f)',
    'SALDO (g) = (e-f)', 'DESPESAS LIQUIDADAS NO BIMESTRE', 'DESPESAS LIQUIDADAS ATÉ O BIMESTRE (h)',
    'SALDO (i) = (e-h)', 'DESPESAS PAGAS ATÉ O BIMESTRE (j)'
]

# Mantendo apenas as colunas especificadas
rreo_df = rreo_df[colunas_ordenadas]

In [ ]:
contas_receitas_correntes = [
    'ReceitaDeContribuicoes',
    'ReceitaPatrimonial',
    'ReceitaDeServicos',
    'OutrasReceitasCorrentes',
    'ReceitasIntraOrcamentariasTotal',
    'RREO3TransferenciasCorrentes',
    'ReceitaTributariaLiquidaExcetoTransferenciasEFUNDEB',
]

contas_receitas_capital = [
    'ReceitasDeOperacoesDeCredito',
    'AlienacaoDeBens',
    'AmortizacoesDeEmprestimos',
    'TransferenciasDeCapital',
    'OutrasReceitasDeCapital'
    
]

filtro_receitas_correntes = rreo_df[rreo_df['cod_conta'].isin(contas_receitas_correntes)]
filtro_receitas_capital = rreo_df[rreo_df['cod_conta'].isin(contas_receitas_capital)]


# Agrupando por 'exercicio' e somando os valores
receitas_correntes_anexo01 = filtro_receitas_correntes.query("coluna == 'Até o Bimestre (c)'").groupby(['exercicio','conta'])['valor'].sum().reset_index()
receitas_correntes_anexo03 = filtro_receitas_correntes.query("coluna == 'TOTAL (ÚLTIMOS 12 MESES)'").groupby(['exercicio','conta'])['valor'].sum().reset_index()

receitas_capital = filtro_receitas_capital.query("coluna == 'Até o Bimestre (c)'").groupby(['exercicio','conta'])['valor'].sum().reset_index()

receitas_brutas = pd.concat([receitas_correntes_anexo01, receitas_correntes_anexo03, receitas_capital]).groupby(['exercicio'])['valor'].sum().reset_index()

receitas_brutas['valor_bilhoes'] = (receitas_brutas['valor'] / 1000000000).round(2)

receitas_brutas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração do estilo do gráfico
sns.set_style()  # Fundo mais limpo
plt.figure(figsize=(12, 5))  # Ajustando o tamanho

# Criando o gráfico de barras
cores = sns.color_palette("Set2", len(receitas_brutas))  # Paleta de cores
plt.bar(receitas_brutas['exercicio'], receitas_brutas['valor_bilhoes'], color=cores)

# Adicionando rótulos e título
plt.xlabel("Exercício", fontsize=12)
plt.ylabel("Valor (Bilhões)", fontsize=12)
plt.title("Receita Bruta Arrecadada (em Bilhões)", fontsize=14, fontweight="bold")

# Ajustando os rótulos do eixo X
plt.xticks(receitas_brutas['exercicio'], rotation=45)

# Adicionando rótulos nas barras
for i, valor in enumerate(receitas_brutas['valor_bilhoes']):
    plt.text(receitas_brutas['exercicio'][i], valor + 0.05, f"{valor:.2f}", 
             ha='center', fontsize=10, fontweight='bold', color='black')

plt.savefig('receita_bruta_arrecadada.png')

# Exibindo o gráfico
plt.show()

